# Importation et chargement des donnees

In [51]:
# Importation des librairies
import pandas as pd
import numpy as np
from datetime import datetime

# model_selection : le module de sklearn qui permet de découper, tester et optimiser les modèles.
from sklearn.model_selection import train_test_split

#  les transformateurs sont des objets utilisés dans les pipelines pour transformer les données
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer    # Remplace les valeurs manquantes par une valeur calculée ou définie.
from sklearn.compose import ColumnTransformer  # permet d’appliquer différentes transformations à différentes colonnes
from sklearn.pipeline import Pipeline    # permet de chaîner plusieurs étapes de traitement et de modélisation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. LOGIQUE DE TRANSFORMATION
# ==========================================

# Fonction pour la conversion des montants
def transform_montant(X):
    """Convertit montant en Gdes.
    - Si contient 'US' => valeur USD * 130
    - Sinon si contient k => valeur milier*1000
    Sinon => valeur deja en Gdes
    - Valeurs non convertibles => NaN
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Transformer en string, strip + uppercase
    s_str = s.astype(str).str.strip()
    s_up = s_str.str.upper()

    # Detecter USD
    is_usd = s_up.str.contains("US", na=False)

    # Detecter k (miliers)
    is_k = s_up.str.contains("K", na=False)

    # Retirer "US", ',', 'K' sans regex, puis nettoyer les espaces
    s_clean = (
        s_up.str.replace("USD", "", regex=False)
        .str.replace("K", "", regex=False)
        .str.replace("HTG", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    # Convertir en float
    num = pd.to_numeric(s_clean, errors="coerce")

    # Conversion: USD -> Gdes * 130, sinon garder tel quel
    out = num.where(~is_usd, num * 130)

    # Conversion: k -> *1000
    out = out.where(~is_k, out * 1000)

    return pd.DataFrame(out, columns=[col])


# Fonction pour le nettoyage des dates
def transform_date_transaction(X):
    """
    Transforme la chaine de str en objet datetime Python
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    if np.issubdtype(s.dtype, np.datetime64):
        dt = s
    else:
        s = s.astype(str).str.strip()
        dt = pd.to_datetime(s, format='%d-%m-%Y', errors='coerce')

    return dt


# Fonction pour le nettoyage de la devise
def transform_device(X):
    """
    Nettoyer la colonne Devise en remplaçant USD par HTG
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    
    s_clean = (
        X[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("USD", "HTG")
    )
    
    return pd.DataFrame(s_clean, columns=[col])


# Fonction pour convertir l'anciennete en jours
def transform_anciennete_en_jours(X):
    """
    Convertir l'anciennete en jours
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col].astype(str).str.strip().str.lower()

    # Detection des unites
    is_years = (
        s.str.contains('year', na=False) |
        s.str.contains('years', na=False)
    )

    is_months = (
        s.str.contains('month', na=False) |
        s.str.contains('months', na=False) |
        s.str.contains('m', na=False)
    )

    # Nettoyage pour ne garder que le chiffre
    # On retire toutes les lettres et on gere la virgule
    s_clean = (
        s.str.replace(r'[a-z]', '', regex=True)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    num = pd.to_numeric(s_clean, errors='coerce')

    # Application des coefficients de conversion
    # np.select(conditions, choix, valeur_par_defaut)
    day = np.select(
        condlist=[is_years, is_months],
        choicelist=[num * 365, num * 30],
        default=num
    )

    return pd.DataFrame(day, columns=[col])


# Fonction pour la normalisation des villes
def normaliser_ville(X):
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    
    # Nettoyage de base
    s_clean = X[col].astype(str).str.strip().str.lower()

    # Dictionnaire de normalisation
    mapping = {
        'p-au-p': 'Port-au-Prince',
        'port au prince': 'Port-au-Prince',
        'pap': 'Port-au-Prince',
        'port-au-prince': 'Port-au-Prince',
        'jacmel': 'Jacmel',
        'cap': 'Cap-Haïtien',
        'cap haitien': 'Cap-Haïtien',
        'cap-haïtien': 'Cap-Haïtien',
        'hin': 'Hinche',
        'hinche': 'Hinche',
        'gonaives': 'Gonaïves',
        'gonaïves': 'Gonaïves',
        'cayes': 'Les Cayes',
        'les cayes': 'Les Cayes'
    }

    # Application du remplacement
    s_clean = s_clean.replace(mapping)

    
    # On remplace les chaines vides ou 'nan' par 'Inconnu'
    s_clean = s_clean.replace(['', 'nan', 'n/a', 'none'], 'Inconnu')
    
    # Mes les premieres lettres en majuscules
    s_clean = s_clean.str.title()

    return pd.DataFrame(s_clean, columns=[col])


# Fonction pour le nettoyage de l'age
def transform_age(X):
    """
    Nettoyer la colonne age en convertissant en numerique et en gerant les valeurs aberrantes    
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base
    s_clean = s.astype(str).str.strip()
    num = pd.to_numeric(s_clean, errors='coerce')

    # je vais clipper les ages entre 18 et 90 ans
    num = num.clip(lower=18, upper=90)

    return pd.DataFrame(num, columns=[col])


# Fonction pour le nettoyage du revenu
def transform_revenu_or_dette(X):
    """
    Nettoyer la colonne RevenuMensuel_raw ou Dette_raw 
    en convertissant en numerique et en gerant les valeurs manquantes    
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base et remplace les virgules
    s_clean = (
        s.astype(str).str.strip()
        .str.replace(',', '', regex=False)
    )

    # Convertir en float
    num = pd.to_numeric(s_clean, errors='coerce')

    return pd.DataFrame(num, columns=[col])


# Fonction pour la normalisation de Employe
def normaliser_employe(X):
    """
    Normaliser la colonne Employe en gerant les valeurs manquantes
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base
    s_clean = s.astype(str).str.strip().str.lower()

    # Dictionnaire de normalisation
    mapping = {
        'yes': 'Oui',
        'no': 'Non',
    }

    # Application du remplacement
    s_clean = s_clean.replace(mapping)

    # On remplace les chaines vides ou 'nan' par 'Inconnu'
    s_clean = s_clean.replace(['', 'nan', 'n/a', 'none'], 'Inconnu')

    # Mettre en majuscule la premiere lettre
    s_clean = s_clean.str.title()

    # Remplace les oui par 1 et les non par 0
    s_clean = s_clean.map({'Oui': 1, 'Non': 0, 'Inconnu': np.nan})

    # Convertir en float
    s_clean_float = pd.to_numeric(s_clean, errors='coerce')

    return pd.DataFrame(s_clean_float, columns=[col])


# Fonction pour traiter les Device
def normaliser_other(X):
    """
    Normaliser la colonne Device en gerant les valeurs manquantes
    """
    X = pd.DataFrame(X).copy()
    col = X.columns[0]
    s = X[col]

    # Nettoyage de base
    s_clean = s.astype("string").str.strip().str.lower()

    # On remplace les chaines vides ou 'nan' par 'Inconnu'
    s_clean = s_clean.replace(['', 'nan', 'n/a', 'none'], 'Inconnu')

    # Mettre en majuscule la premiere lettre
    s_clean = s_clean.str.title()

    return pd.DataFrame(s_clean, columns=[col])


# ==========================================
# 2. CHARGEMENT ET PREPARATION
# ==========================================

# Chargements des donnees
df = pd.read_excel('dataset_pretraitement_fraude.xlsx')

# Suppression les doublons
df = df.drop_duplicates()

X = df.drop(
    columns=[
        'Fraude',
        'TransactionID',
        'ClientID',
        'Commentaire',
        'DateTransaction_raw',
        'TODO_Montant_HTG',
        'TODO_Revenu_clean',
        'TODO_Dette_clean',
        'TODO_Anciennete_jours',
        'TODO_DateTransaction',
        'TODO_JourSemaine',
        'TODO_Age_clean',
        'TODO_Ville_norm',
    ]
)

y = df['Fraude']


# ==========================================
# 3. PIPELINE DE PRETRAITEMENT + MODELE
# ==========================================

# On definit les colonnes pour eviter de se perdre
preprocessing_pipeline = ColumnTransformer(
    transformers=[
        # 1. Montant : Transform, Impute, Scale
        ('num_montant', Pipeline([
            ('convert', FunctionTransformer(transform_montant)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['Montant_raw']),

        # 2. Anciennete : Transform, Impute, Scale
        ('num_anciennete', Pipeline([
            ('convert', FunctionTransformer(transform_anciennete_en_jours)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['AncienneteCompte_raw']),

        # 3. Revenu & Dette : Transform, Impute, Scale j'utilise la meme fonction pour les deux
        ('num_revenu', Pipeline([
            ('convert', FunctionTransformer(transform_revenu_or_dette)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['RevenuMensuel_raw']),

        ('num_dette', Pipeline([
            ('convert', FunctionTransformer(transform_revenu_or_dette)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['Dette_raw']),

        # 4. Age : Transform, Impute, Scale
        ('num_age', Pipeline([
            ('convert', FunctionTransformer(transform_age)),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['Age']),

        # 5. Employe j'ai deja converti en 0/1/NaN : Impute
        ('num_employe', Pipeline([
            ('convert', FunctionTransformer(normaliser_employe)),
            ('imputer', SimpleImputer(strategy='most_frequent')) # On remplit les 'Inconnu' qui etaient NaN
        ]), ['Employe']),

        # 6. Categories : Normaliser, OneHotEncoder
        ('cat_ville', Pipeline([
            ('norm', FunctionTransformer(normaliser_ville)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Ville_raw']),

        ('cat_autres', Pipeline([
            ('norm', FunctionTransformer(normaliser_other)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Device', 'TypeMarchand', 'NiveauEtude', 'Canal']),

        # 7. NbTrans_24h : Impute, Scale
        ('num_nbtrans_24h', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['NbTrans_24h']),

        # 8. StatutMarital : Impute, OneHotEncoder
        ('cat_statutmarital', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['StatutMarital']),

        # 9. Devise_indiquee : Normaliser, Impute, OneHotEncoder
        ('cat_devise', Pipeline([
            ('norm', FunctionTransformer(transform_device)),
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Devise_indiquee']),
    ]
)

full_model = Pipeline([
    ('pretraitement', preprocessing_pipeline),
    ('modele', LogisticRegression(max_iter=1000))
])

# ==========================================
# 4. ENTRAINEMENT ET EVALUATION
# ==========================================

# Decoupage en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entrainement du modele
full_model.fit(X_train, y_train)

# Prediction sur le jeu de test
y_pred = full_model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.7666666666666667
              precision    recall  f1-score   support

           0       0.78      0.97      0.87        93
           1       0.40      0.07      0.12        27

    accuracy                           0.77       120
   macro avg       0.59      0.52      0.50       120
weighted avg       0.70      0.77      0.70       120



In [43]:
# Test de la fonction anciennete
transform_anciennete_en_jours(X[['AncienneteCompte_raw']].head(20))

,AncienneteCompte_raw
0,1265
1,4015
2,2324
3,2550
4,89
5,365
6,2520
7,645
8,0
9,3229


In [ ]:
# Separer les caracteristik numeriques et categoriels
X_numeriques = [
    'TODO_Montant_HTG',
    'TODO_Revenu_clean',
    'TODO_Dette_clean',
    'TODO_Anciennete_jours',
    'TODO_JourSemaine',
    'TODO_JourMois',
    'TODO_Mois',
    'TODO_Annee',
    'TODO_Age_clean',
    'NbTrans_24h',
]

X_categoriels = [
    'Devise_indiquee',
    'TODO_Ville_norm',
    'Employe',
    'Device',
    'NiveauEtude',
    'StatutMarital',
]

# Kole yo
X = data[X_numeriques + X_categoriels]

# La colonne cible
y = data['Fraude']

# Separation des donnees
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

display(y_train.head())